This is the solution of F21BC Week 4 Lab with keras version code - Part 4

a) Check the code from data_handler.py and convlstm.py and run it.

b) Change the dropout rate on the LSTM layer to 80%.

c) Train the network with only the 6 first features (columns) of the data.

**Task b:** Change the dropout rate on the LSTM layer to 80%.

In [1]:
# -*- coding: utf-8 -*-
#  data_handler.py

import numpy as np
import csv
from sklearn.preprocessing import normalize as norm


def window_data( data, window_size ):
    data = np.array( data )
    w_data = [ data[ i*window_size : (i+1)*window_size ]
               for i in range( len(data) // window_size ) ]
    return w_data


def load_multimodal( split_num,
                     num_activities=20,
                     num_seqs=10,
                     window_size = 10 ):
    train_data   = list()
    test_data    = list()
    train_labels = list()
    test_labels  = list()
    for act_id in range( num_activities ):
        for seq_id in range( num_seqs ):
            act_num = '{:02d}'.format( act_id+1 )
            seq_num = '{:02d}'.format( seq_id+1 )
            with open('data/act' + act_num + 'seq' + seq_num + '.csv', 'r') as f:
                inp = list( csv.reader(f) )
                inps = window_data( inp, window_size )
                # Provide one-hot label
                label = np.zeros( [len(inps), num_activities] )
                label[ :, act_id ] = 1
                label = label.tolist()
                # Split data into train and test sets
                if seq_id+1 == split_num:
                    test_data   += inps
                    test_labels += label
                else:
                    train_data   += inps
                    train_labels += label
    train_data = [ norm(x) for x in train_data ]
    test_data  = [ norm(x) for x in test_data ]
    return train_data, train_labels, test_data, test_labels

if __name__ == '__main__':
    X_train, y_train, X_test, y_test = load_multimodal ( split_num = 1,
                                                         window_size = 10)
    print( np.array(X_train).shape )
    print( np.array(y_train).shape )
    print( np.array(X_test).shape )
    print( np.array(y_test).shape )

(2698, 10, 19)
(2698, 20)
(300, 10, 19)
(300, 20)


In [ ]:
from keras.models import load_model, Model, Sequential
from keras.layers import Input, Dense, Conv1D, MaxPooling1D, LSTM
# import data_handler as data
import numpy as np
import os

# Task b: Change the dropout rate on the LSTM layer to 80%.
def create_model(shape, num_classes):
    model = Sequential()
    model.add( Conv1D(128, 3, padding='same', activation='relu', input_shape=shape) )
    model.add( MaxPooling1D(2) )

    model.add( Conv1D(256, 3, padding='same', activation='relu') )
    model.add( MaxPooling1D(2) )

    model.add( LSTM(128, dropout=0.8, activation = 'relu') )

    model.add( Dense(num_classes, activation='softmax') )
    return model



if __name__ == '__main__':
    batch_size   = 32
    timesteps    = 60

    num_features = 19

    accuracies   = list()
    # 10-fold
    #for fold in range(10):
    for fold in range(1):
        print('initializing fold', fold)
        model = create_model( shape = (timesteps, num_features),
                              num_classes = 20 )
        X_train, y_train, X_test, y_test = load_multimodal ( fold+1,
                                                                  window_size = timesteps)
        X_train, y_train = np.array(X_train), np.array(y_train)
        X_test, y_test = np.array(X_test), np.array(y_test)
        model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

        model.fit( X_train, y_train,
                   batch_size = 32,
                   epochs=150 )
        loss, acc = model.evaluate( X_test, y_test )
        accuracies.append(acc)
        print("Model acc:", acc, "loss:", loss)
    print('mean acc:', sum(accuracies)/len(accuracies))



**Task c**: Train the network with only the 6 first features (columns) of the data.

In [5]:
# -*- coding: utf-8 -*-
#  data_handler.py
# Firstly, modify the 'load_multimodal' function, load only 6 features.

import numpy as np
import csv
from sklearn.preprocessing import normalize as norm


def window_data( data, window_size ):
    data = np.array( data )
    w_data = [ data[ i*window_size : (i+1)*window_size ]
               for i in range( len(data) // window_size ) ]
    return w_data


def load_multimodal( split_num,
                     num_activities=20,
                     num_seqs=10,
                     window_size = 10 ):
    train_data   = list()
    test_data    = list()
    train_labels = list()
    test_labels  = list()
    for act_id in range( num_activities ):
        for seq_id in range( num_seqs ):
            act_num = '{:02d}'.format( act_id+1 )
            seq_num = '{:02d}'.format( seq_id+1 )
            with open('data/act' + act_num + 'seq' + seq_num + '.csv', 'r') as f:
                inp = list( csv.reader(f) )

                # Task b: load the first 6 features
                inp = np.array(inp, dtype=float)[:, :6]

                inps = window_data( inp, window_size )
                # Provide one-hot label
                label = np.zeros( [len(inps), num_activities] )
                label[ :, act_id ] = 1
                label = label.tolist()
                # Split data into train and test sets
                if seq_id+1 == split_num:
                    test_data   += inps
                    test_labels += label
                else:
                    train_data   += inps
                    train_labels += label
    train_data = [ norm(x) for x in train_data ]
    test_data  = [ norm(x) for x in test_data ]
    return train_data, train_labels, test_data, test_labels

if __name__ == '__main__':
    X_train, y_train, X_test, y_test = load_multimodal ( split_num = 1,
                                                         window_size = 10)
    print( np.array(X_train).shape )
    print( np.array(y_train).shape )
    print( np.array(X_test).shape )
    print( np.array(y_test).shape )

(2698, 10, 6)
(2698, 20)
(300, 10, 6)
(300, 20)


In [ ]:
from keras.models import load_model, Model, Sequential
from keras.layers import Input, Dense, Conv1D, MaxPooling1D, LSTM
# import data_handler as data
import numpy as np
import os

# Task b: Change the dropout rate on the LSTM layer to 80%.
def create_model(shape, num_classes):
    model = Sequential()
    model.add( Conv1D(128, 3, padding='same', activation='relu', input_shape=shape) )
    model.add( MaxPooling1D(2) )

    model.add( Conv1D(256, 3, padding='same', activation='relu') )
    model.add( MaxPooling1D(2) )

    model.add( LSTM(128, dropout=0.8, activation = 'relu') )

    model.add( Dense(num_classes, activation='softmax') )
    return model



if __name__ == '__main__':
    batch_size   = 32
    timesteps    = 60

    # task c: Train the network with only the 6 first features (columns) of the data.
    # num_features = 19
    num_features = 6

    accuracies   = list()
    # 10-fold
    #for fold in range(10):
    for fold in range(1):
        print('initializing fold', fold)
        model = create_model( shape = (timesteps, num_features),
                              num_classes = 20 )
        X_train, y_train, X_test, y_test = load_multimodal ( fold+1,
                                                                  window_size = timesteps)
        X_train, y_train = np.array(X_train), np.array(y_train)
        X_test, y_test = np.array(X_test), np.array(y_test)
        model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

        model.fit( X_train, y_train,
                   batch_size = 32,
                   epochs=150 )
        loss, acc = model.evaluate( X_test, y_test )
        accuracies.append(acc)
        print("Model acc:", acc, "loss:", loss)
    print('mean acc:', sum(accuracies)/len(accuracies))
